### Hypergraph MaxCut with GNN-Based Model
This example demonstrates solving MaxCut on a synthetic hypergraph using a GNN-based model (with GraphSAGE layers) via the run_pubo framework.

In [2]:
import sys
from pathlib import Path
current = Path.cwd()
root_path = current.parents[3]  
sys.path.append(str(root_path))
import torch
import pandas as pd
from src import generate_data
from src.maxcut import Net
from src import init, get_device, run_pubo, Layer, LayerType

### Initialization and Data Preparation

In [ ]:
init(cuda_index=0, reproducibility=False)  
n = 130
e = n * 4
data = generate_data("hypergraph", n, e).to(get_device())
init_feature_dim = 2048
num_group = 2 # max-cut 
x = torch.rand((data.num_v, init_feature_dim))
layers = [
    Layer(LayerType.GRAPHSAGE, init_feature_dim, 256, hidden_channels=512, num_layers=2, jk="last", drop_rate=0, act="leaky_relu"),
    Layer(LayerType.LINEAR, 256, num_group, use_bn=True, drop_rate=0.2),
]
net = Net(layers)

### Define Dynamic Weight Scheduling and Run Optimization

In [ ]:


gini_cof_lambda = lambda e, n: (-200 + 0.25 * e) / 10
# gini_cof_lambda = lambda e, n: 0
loss, outs, eval = run_pubo(
    'maxcut',
    net,
    x,
    data,
    2200,
    2.5e-5,
    opt="AdamW",
    gini_cof_lambda=gini_cof_lambda,
    clip_grad=True,
    evaluate=True
)

[INFO] Using CUDA device: NVIDIA GeForce RTX 3090 (Index: 0)
[INFO] INFO: 0 loops found
[INFO] Hypergraph to graph: 130 vertices 1245 graph edges
 95%|█████████▍| 2080/2200 [00:09<00:00, 548.77it/s]

Epoch: 2000.00 | obj Loss: -429.00 | gini Loss: 0.00 | 


100%|██████████| 2200/2200 [00:09<00:00, 232.07it/s]

+------------[MaxCut Evaluation]------------+
Cut Edges: 356/520 (68.5%)
Group Distribution: 65 vs 65
Unconverged Nodes: 0
+--------------------------------------------+
